<div style="background:linear-gradient(135deg,#1A5276 0%,#2E86C1 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#AED6F1;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 2 · CLASE 6</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Ejercicios: Evaluación de Regresión Logística</h1>
  <p style="color:#AED6F1;font-size:16px;margin:0 0 24px 0;font-style:italic;">Deviance · AIC · Wald test · Pseudo R² · ROC/AUC</p>
  <hr style="border-color:#5DADE2;margin:0 0 20px 0;">
  <p style="color:#EBF5FB;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; 🟢 Básico · 🟡 Intermedio · 🔴 Avanzado</p>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import expit
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
np.set_printoptions(precision=4,suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)
print('✅ Setup listo')

---
## 🟢 Ejercicio 1 — AUC desde cero

Implementar AUC manualmente usando la definición:

AUC = P(p̂(positivo) > p̂(negativo))

1. Generar modelo logístico y obtener probabilidades
2. Implementar AUC por comparación de pares
3. Verificar contra sklearn
4. Graficar curva ROC

In [ ]:
np.random.seed(SEED)
n=400; X_r=np.random.randn(n,3)
beta_r=np.array([-0.5,1.2,-0.8,0.5])
X_full=np.column_stack([np.ones(n),X_r])
y_r=np.random.binomial(1,expit(X_full@beta_r))
sc=StandardScaler(); Xsc=sc.fit_transform(X_r)
Xtr,Xte,ytr,yte=train_test_split(Xsc,y_r,test_size=0.25,random_state=SEED)
lr=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500)
lr.fit(Xtr,ytr); proba=lr.predict_proba(Xte)[:,1]
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
n=400; X_r=np.random.randn(n,3); beta_r=np.array([-0.5,1.2,-0.8,0.5])
X_full=np.column_stack([np.ones(n),X_r]); y_r=np.random.binomial(1,expit(X_full@beta_r))
sc=StandardScaler(); Xsc=sc.fit_transform(X_r)
Xtr,Xte,ytr,yte=train_test_split(Xsc,y_r,test_size=0.25,random_state=SEED)
lr=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500); lr.fit(Xtr,ytr)
proba=lr.predict_proba(Xte)[:,1]

# AUC manual
pos=proba[yte==1]; neg=proba[yte==0]
comparaciones=np.array([[p>q for q in neg] for p in pos])
auc_manual=comparaciones.mean()
auc_sklearn=roc_auc_score(yte,proba)
print(f'AUC manual:  {auc_manual:.6f}')
print(f'AUC sklearn: {auc_sklearn:.6f}')
print(f'¿Coinciden? {np.isclose(auc_manual,auc_sklearn,atol=0.01)}')
fpr,tpr,_=roc_curve(yte,proba)
fig,ax=plt.subplots(figsize=(6,5))
ax.plot(fpr,tpr,color='#2E86C1',lw=2,label=f'AUC={auc_sklearn:.4f}')
ax.plot([0,1],[0,1],'k--',lw=1); ax.fill_between(fpr,tpr,alpha=0.15,color='#2E86C1')
ax.set(xlabel='FPR',ylabel='TPR',title='Curva ROC'); ax.legend(); ax.grid(True,alpha=0.2)
plt.tight_layout(); plt.show()

---
## 🟡 Ejercicio 2 — Comparar modelos con AIC

1. Ajustar 3 modelos con distintos subsets de features
2. Calcular AIC de cada uno
3. ¿Cuál es el mejor? ¿Coincide con el que tiene más features?

In [ ]:
np.random.seed(SEED)
n=500; X_r2=np.random.randn(n,5)
# Solo x1,x2,x3 tienen efecto real
beta_aic=np.array([-0.8,1.5,-1.2,0.9,0.0,0.0])
Xf2=np.column_stack([np.ones(n),X_r2])
y_aic=np.random.binomial(1,expit(Xf2@beta_aic))

subsets=[(0,1,2,3),(0,1,2),(0,1,2,3,4,5),(0,)]
nombres=['x1,x2,x3 (real)','x1,x2','x1-x5 (todos)','solo intercepto']
# --- Calcular AIC para cada subset ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED); n=500; X_r2=np.random.randn(n,5)
beta_aic=np.array([-0.8,1.5,-1.2,0.9,0.0,0.0])
Xf2=np.column_stack([np.ones(n),X_r2]); y_aic=np.random.binomial(1,expit(Xf2@beta_aic))
subsets=[(0,1,2,3),(0,1,2),(0,1,2,3,4,5),(0,)]
nombres=['x1,x2,x3','x1,x2','x1-x5','intercepto']
print(f'{"Modelo":20s} {"p":>4s} {"ℓ":>12s} {"AIC":>10s} {"Mejor"}')
print('─'*55)
AICs=[]
for cols,nm in zip(subsets,nombres):
    Xc=Xf2[:,list(cols)]; p=len(cols)
    def neg_ll(b): return -np.sum(y_aic*np.log(np.clip(expit(Xc@b),1e-10,1-1e-10))+(1-y_aic)*np.log(np.clip(1-expit(Xc@b),1e-10,1-1e-10)))/n
    def gr(b): return -Xc.T@(y_aic-expit(Xc@b))/n
    res=minimize(neg_ll,np.zeros(p),method='BFGS',jac=gr)
    ll=(-res.fun)*n; aic=2*p-2*ll; AICs.append(aic)
    print(f'  {nm:18s} {p:>4d} {ll:>12.2f} {aic:>10.2f}')
best=nombres[np.argmin(AICs)]
print(f'\n→ Mejor modelo: {best} (menor AIC)')

---
## 🔴 Ejercicio 3 — Reporte completo de logística

Implementar `logistic_report(X, y, feat_names)` que genere:
1. Pseudo R² · AIC · Deviance
2. Tabla de coeficientes con Wald test + OR + IC95%
3. Curva ROC + AUC
4. Threshold óptimo (maximize Youden J = sensib + especif - 1)

In [ ]:
# --- Tu implementación aquí ---
def logistic_report(X, y, feat_names=None):
    pass

np.random.seed(SEED); n=400
X_demo=StandardScaler().fit_transform(np.random.randn(n,3))
y_demo=np.random.binomial(1,expit(np.column_stack([np.ones(n),X_demo])@np.array([-1,1.5,-0.8,0.5])))
logistic_report(X_demo, y_demo, feat_names=['x1','x2','x3'])

In [ ]:
# ✅ SOLUCIÓN
def logistic_report(X,y,feat_names=None):
    n,p_feat=X.shape; names=feat_names or [f'x{i}' for i in range(p_feat)]
    Xf=np.column_stack([np.ones(n),X]); p_total=Xf.shape[1]
    def neg_ll(b): return -np.sum(y*np.log(np.clip(expit(Xf@b),1e-10,1-1e-10))+(1-y)*np.log(np.clip(1-expit(Xf@b),1e-10,1-1e-10)))/n
    def gr(b): return -Xf.T@(y-expit(Xf@b))/n
    res=minimize(neg_ll,np.zeros(p_total),method='BFGS',jac=gr)
    beta=res.x; ll=(-res.fun)*n
    p_hat=expit(Xf@beta); W=p_hat*(1-p_hat)
    I=Xf.T@(W[:,None]*Xf); SE=np.sqrt(np.diag(np.linalg.inv(I)))
    z=beta/SE; pv=2*(1-stats.norm.cdf(np.abs(z)))
    ll_null=np.sum(y*np.log(y.mean())+(1-y)*np.log(1-y.mean()))
    R2_mcf=1-ll/ll_null; AIC=2*p_total-2*ll; D=-2*ll
    print('═'*65); print('  REPORTE DE REGRESIÓN LOGÍSTICA'); print('═'*65)
    print(f'  n={n}  p={p_total}  R²_McF={R2_mcf:.4f}  AIC={AIC:.2f}  Deviance={D:.2f}')
    print('─'*65)
    print(f'  {"Feature":14s} {"β̂":>8s} {"SE":>8s} {"z":>8s} {"p-val":>9s} {"OR":>8s} {"Sig"}')
    print('─'*65)
    for i,nm in enumerate(['intercepto']+names):
        sig='***' if pv[i]<0.001 else '**' if pv[i]<0.01 else '*' if pv[i]<0.05 else '(ns)'
        OR=np.exp(beta[i]) if i>0 else np.nan
        print(f'  {nm:14s} {beta[i]:>8.4f} {SE[i]:>8.4f} {z[i]:>8.3f} {pv[i]:>9.4f} {"-" if np.isnan(OR) else f"{OR:.4f}":>8s} {sig}')
    Xtr_,Xte_,ytr_,yte_=train_test_split(X,y,test_size=0.25,random_state=SEED)
    lr_=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500); lr_.fit(Xtr_,ytr_)
    pr_=lr_.predict_proba(Xte_)[:,1]
    auc=roc_auc_score(yte_,pr_); fpr_,tpr_,thr=roc_curve(yte_,pr_)
    J=tpr_-fpr_; opt=np.argmax(J)
    print(f'\n  AUC={auc:.4f}  Threshold óptimo={thr[opt]:.3f}  (Youden J={J[opt]:.3f})')
    print('═'*65)
    fig,ax=plt.subplots(figsize=(6,4))
    ax.plot(fpr_,tpr_,color='#2E86C1',lw=2,label=f'AUC={auc:.4f}')
    ax.scatter(fpr_[opt],tpr_[opt],color='red',s=80,zorder=5,label=f'Óptimo thr={thr[opt]:.2f}')
    ax.plot([0,1],[0,1],'k--',lw=1); ax.set(xlabel='FPR',ylabel='TPR',title='ROC')
    ax.legend(); ax.grid(True,alpha=0.2); plt.tight_layout(); plt.show()

np.random.seed(SEED); n=400
X_demo=StandardScaler().fit_transform(np.random.randn(n,3))
y_demo=np.random.binomial(1,expit(np.column_stack([np.ones(n),X_demo])@np.array([-1,1.5,-0.8,0.5])))
logistic_report(X_demo,y_demo,feat_names=['x1','x2','x3'])

---
<div style="background:#1A5276;color:white;padding:20px 24px;border-radius:8px;">
<strong>Próxima clase — Clase 7</strong><br>
Threshold óptimo · Caso de scoring de crédito completo · Fin Módulo 2<br>
<em>Docente: Josef Rodriguez · Curso 8 · Módulo 2</em>
</div>